# 06v2 - A2: pooled 4-fold CV

**Follows 05v2** (A2 v1, gold macro-AUC 0.7689 on fold 0 alone). That run
used only 1 of 4 folds; this notebook completes the other 3 folds and
pools every fold's held-out gold predictions into one macro-AUC over all
58 gold studies, instead of the 17-gold subset of a single fold. Two
motivations, both from the fold-0-only result: `medial_meniscus_tear`
scored 0.458 (below random) and `oa_lateral_compartment` was undefined
(single class among only 17 gold studies) - both are exactly the kind of
small-sample artifact a pooled 58-study OOF number resolves.

Self-contained per this project's Kaggle constraint (no `import src`,
confirmed 2026-08-26) - same hand-kept-copy pattern as 05v2, kept in sync
manually with `src/data.py`/`src/model.py`/`src/evaluate.py`.

**Design:**
- **Fold 0**: reuses the existing checkpoint (`a2_v1_fold0_best.pt`,
  trained in 05v2) for **inference only** - no retraining. That checkpoint
  never saw fold 0's studies during training, so scoring it on fold 0's
  val set again is a valid, cheap (~seconds) OOF read, not leakage.
  Requires the checkpoint attached as a Kaggle Dataset input (see the
  `CHECKPOINT_PATH` constant below - same "upload it yourself, edit the
  path" convention as `PUBLISHED_LABELS_PATH`).
- **Folds 1, 2, 3**: trained fresh in this same session, same
  architecture/hyperparameters as 05v2 (fixed here: `BATCH_SIZE=32`,
  matching the real value 05v2 actually ran with, not the `8` its
  committed code had at the time - see that notebook's fold-0 cell for
  the correction).
- A single pre-flight (not repeated per fold) validates the training
  function once before spending real GPU time on 3 runs.
- Real cost: ~47 min/fold x 3 = **~2.5h of GPU**, plus the fold-0
  inference pass (seconds).

**Correctness check:** `GroupKFold` has no shuffling/random state, so
recomputing the same fold assignment from the same inputs in this
notebook should reproduce 05v2's exact fold 0 (1,307 val studies, 17
gold) - asserted below before trusting the reused checkpoint's
predictions.

In [ ]:
import hashlib
import re
import time
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd
import pydicom
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupKFold

_KAGGLE_RAW = Path("/kaggle/input/competitions/rsna-knee-abnormality-detection")
ON_KAGGLE = _KAGGLE_RAW.exists()
if not ON_KAGGLE:
    raise RuntimeError(
        "This notebook needs the full DICOM tree + GPU + the A3 cache "
        "attached - run on Kaggle, not locally."
    )
RAW_DIR = _KAGGLE_RAW
CACHE_DIR = Path("/kaggle/input/datasets/alherma7/cache-stevenleehans-rsna/cache")

# Same convention as 05v2's PUBLISHED_LABELS_PATH: not part of the
# official competition mount, a separate small Dataset the user attaches
# themselves. EDIT this path to match wherever it actually lands under
# /kaggle/input/ once attached (check with `!ls /kaggle/input` if unsure).
PUBLISHED_LABELS_PATH = Path("/kaggle/input/llm-labels-v4-blend/llm_labels_v4_blend.csv")

# The 05v2 fold-0 checkpoint (models/a2_v1_fold0_best.pt locally,
# gitignored) - upload it as a Kaggle Dataset and attach it to this
# kernel, then EDIT this path to match. Used for inference only (no
# retraining) in the "Fold 0" section below.
CHECKPOINT_PATH = Path("/kaggle/input/a2-v1-fold0-checkpoint/a2_v1_fold0_best.pt")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("DEVICE:", DEVICE)
print("CACHE_DIR:", CACHE_DIR, "exists:", CACHE_DIR.exists())
print("PUBLISHED_LABELS_PATH:", PUBLISHED_LABELS_PATH, "exists:", PUBLISHED_LABELS_PATH.exists())
print("CHECKPOINT_PATH:", CHECKPOINT_PATH, "exists:", CHECKPOINT_PATH.exists())

FINDINGS = [
    "acl_injury", "mcl_injury", "medial_meniscus_tear", "lateral_meniscus_tear",
    "oa_medial_compartment", "oa_lateral_compartment", "oa_patellofemoral_compartment",
    "effusion", "synovitis", "bakers_cyst", "bone_contusion", "fracture",
]
OFFICIAL_LABEL_COLUMNS = {
    "acl_injury": "ACL", "mcl_injury": "MCL",
    "medial_meniscus_tear": "Medial Meniscus", "lateral_meniscus_tear": "Lateral Meniscus",
    "oa_medial_compartment": "Medial OA", "oa_lateral_compartment": "Lateral OA",
    "oa_patellofemoral_compartment": "PF OA", "effusion": "Effusion",
    "synovitis": "Synovitis", "bakers_cyst": "Baker's",
    "bone_contusion": "Contusion", "fracture": "Fracture",
}
SLOT_NAMES = ["SAG_FLUID_FS", "COR_FLUID_FS", "AX_FLUID_FS", "SAG_FLUID_NOFS", "COR_T1", "SAG_T1"]
SLOT_CACHE_GROUP_SIZE = 3
GROUP_INDEX = 1  # centre anchor, per the approved A2 spec section 1
CV_FOLDS = 4
TRAIN_SHARDS = [f"train.s{i:02d}of04" for i in range(4)]
BATCH_SIZE = 32  # matches 05v2's real run (see that notebook's fold-0 cell)
EPOCHS = 12

## Labels: gold official values + A1a' published set for weak studies

In [ ]:
def load_published_labels(path):
    published = pd.read_csv(path)
    label_cols = list(OFFICIAL_LABEL_COLUMNS.values())
    published = published.set_index("StudyInstanceUID")[label_cols]
    published.columns = list(OFFICIAL_LABEL_COLUMNS.keys())
    return published


def load_gold_labels(raw_dir):
    train = pd.read_csv(raw_dir / "train.csv")
    label_cols = list(OFFICIAL_LABEL_COLUMNS.values())
    gold_mask = train[label_cols].notna().all(axis=1)
    gold = train.loc[gold_mask, ["StudyInstanceUID"] + label_cols].set_index("StudyInstanceUID")
    gold.columns = list(OFFICIAL_LABEL_COLUMNS.keys())
    return gold


train_csv = pd.read_csv(RAW_DIR / "train.csv")
reports = train_csv.set_index("StudyInstanceUID")[["Report"]]
gold = load_gold_labels(RAW_DIR)
published = load_published_labels(PUBLISHED_LABELS_PATH)

missing = set(train_csv["StudyInstanceUID"]) - set(published.index)
print("train.csv studies missing from published labels:", len(missing))
assert len(missing) == 0

is_gold = reports.index.isin(gold.index)
label_table = published.reindex(reports.index)[FINDINGS].copy()
label_table.loc[gold.index, FINDINGS] = gold[FINDINGS]
label_table["is_gold"] = is_gold
print(label_table.shape, "gold rows:", label_table["is_gold"].sum())
assert label_table["is_gold"].sum() == 58

## Folds: report-template + scanner-fingerprint grouping (A0)

Identical logic to 05v2's fold-assignment cell. `GroupKFold` has no
shuffling or random state, so given the same `train.csv` row order and
the same `group_ids`, this reproduces 05v2's exact fold assignment -
verified below against that run's real recorded numbers (1,307 val / 17
gold in fold 0) before trusting the reused checkpoint's predictions.

In [ ]:
def report_group_key(report_text):
    if not isinstance(report_text, str):
        normalized = ""
    else:
        t = unicodedata.normalize("NFKD", report_text.lower())
        t = "".join(ch for ch in t if not unicodedata.combining(ch))
        normalized = re.sub(r"\s+", " ", t).strip()
    return hashlib.sha256(normalized.encode("utf-8")).hexdigest()


SCANNER_FINGERPRINT_TAGS = (
    "Manufacturer", "ManufacturerModelName", "InstitutionName",
    "DeviceSerialNumber", "MagneticFieldStrength", "StationName",
)


def build_scanner_fingerprints(raw_dir, split="train"):
    series = pd.read_csv(raw_dir / f"{split}_series.csv")
    first_series = series.drop_duplicates("StudyInstanceUID", keep="first")
    fingerprints = {}
    for row in first_series.itertuples(index=False):
        series_dir = raw_dir / f"{split}_series" / row.StudyInstanceUID / row.SeriesInstanceUID
        files = sorted(series_dir.glob("*.dcm"))
        if not files:
            fingerprints[row.StudyInstanceUID] = None
            continue
        ds = pydicom.dcmread(files[0], stop_before_pixels=True)
        fingerprints[row.StudyInstanceUID] = tuple(
            str(getattr(ds, tag, None)) for tag in SCANNER_FINGERPRINT_TAGS
        )
    result = pd.Series(fingerprints, name="scanner_fingerprint")
    result.index.name = "StudyInstanceUID"
    return result


def build_group_ids(*group_key_series):
    index = group_key_series[0].index
    parent = {i: i for i in index}

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    def union(a, b):
        ra, rb = find(a), find(b)
        if ra != rb:
            parent[ra] = rb

    for keys in group_key_series:
        valid = keys.dropna()
        for _, idx in valid.groupby(valid).groups.items():
            idx = list(idx)
            for other in idx[1:]:
                union(idx[0], other)

    return pd.Series({i: find(i) for i in index}, name="group_id")


t0 = time.time()
scanner_fp = build_scanner_fingerprints(RAW_DIR, split="train")
print(f"scanner fingerprints: {time.time() - t0:.1f}s for {len(scanner_fp)} studies")

group_keys = reports["Report"].apply(report_group_key)
group_ids = build_group_ids(group_keys, scanner_fp.reindex(reports.index))

gkf = GroupKFold(n_splits=CV_FOLDS)
fold = pd.Series(-1, index=reports.index, dtype=int)
for fold_idx, (_, val_idx) in enumerate(gkf.split(reports, groups=group_ids.to_numpy())):
    fold.iloc[val_idx] = fold_idx
label_table["fold"] = fold
print(label_table["fold"].value_counts().sort_index())

label_table.to_csv("/kaggle/working/fold_assignments.csv")
print("saved fold_assignments.csv")

# Consistency check against 05v2's real recorded fold-0 output - if this
# doesn't match, GroupKFold produced a different split than the fold-0
# checkpoint was trained against, and reusing it below would be invalid.
fold0_val = label_table[label_table["fold"] == 0]
print(f"fold 0: {len(fold0_val)} val ({fold0_val['is_gold'].sum()} gold)")
assert len(fold0_val) == 1307, f"expected 1307 val studies in fold 0, got {len(fold0_val)}"
assert fold0_val["is_gold"].sum() == 17, f"expected 17 gold in fold 0, got {fold0_val['is_gold'].sum()}"
print("fold assignment matches 05v2's real run - safe to reuse the fold-0 checkpoint")

## Cache dataset

Identical to 05v2 - opens all 4 train shards as memmaps, `group_index=1`
selects the centre anchor's 3 slices.

In [ ]:
class SlotCacheDataset(torch.utils.data.Dataset):
    def __init__(self, cache_dir, shards, labels_df, group_index=GROUP_INDEX, study_ids=None):
        '''study_ids: optional subset to restrict this dataset to (e.g. the
        8 studies of a pre-flight smoke test) - shards are always opened in
        full (cheap, memmap only), then filtered down to this subset before
        the labels_df coverage check below, so labels_df only needs to
        cover the subset, not every study in the loaded shards.'''
        self.group_index = group_index
        caches, masks, all_study_ids, shard_of, local_idx = [], [], [], [], []
        for shard in shards:
            cache = np.load(cache_dir / f"{shard}_cache.npy", mmap_mode="r")
            mask = np.load(cache_dir / f"{shard}_mask.npy")
            studies = pd.read_csv(cache_dir / f"{shard}_studies.csv")
            caches.append(cache)
            masks.append(mask)
            all_study_ids.append(studies["StudyInstanceUID"].to_numpy())
            shard_of.append(np.full(len(studies), len(caches) - 1))
            local_idx.append(np.arange(len(studies)))

        self.caches = caches
        mask_all = np.concatenate(masks, axis=0).astype(np.float32)
        study_ids_all = np.concatenate(all_study_ids)
        shard_of_all = np.concatenate(shard_of)
        local_idx_all = np.concatenate(local_idx)

        if study_ids is not None:
            keep = np.isin(study_ids_all, np.asarray(list(study_ids)))
            mask_all, study_ids_all = mask_all[keep], study_ids_all[keep]
            shard_of_all, local_idx_all = shard_of_all[keep], local_idx_all[keep]

        self.mask = mask_all
        self.study_ids = study_ids_all
        self.shard_of = shard_of_all
        self.local_idx = local_idx_all

        aligned = labels_df.reindex(self.study_ids)[FINDINGS]
        if aligned.isna().any().any():
            missing = self.study_ids[aligned.isna().any(axis=1).to_numpy()]
            raise ValueError(f"{len(missing)} cache studies missing labels, e.g. {missing[:5]}")
        self.labels = aligned.to_numpy(dtype=np.float32)

    def __len__(self):
        return len(self.study_ids)

    def __getitem__(self, i):
        shard_idx, row = self.shard_of[i], self.local_idx[i]
        full = self.caches[shard_idx][row]  # (6, 9, 224, 224) uint8
        g = self.group_index
        selected = full[:, g * SLOT_CACHE_GROUP_SIZE:(g + 1) * SLOT_CACHE_GROUP_SIZE]
        images = torch.from_numpy(np.ascontiguousarray(selected)).float() / 255.0
        mask = torch.from_numpy(self.mask[i])
        label = torch.from_numpy(self.labels[i])
        return images, mask, label


sanity_ds = SlotCacheDataset(CACHE_DIR, TRAIN_SHARDS[:1], label_table)
images, mask, label = sanity_ds[0]
print("images:", images.shape, images.dtype, "mask:", mask.shape, "label:", label.shape)
assert images.shape == (6, 3, 224, 224)
assert not torch.isnan(images).any()
print("SlotCacheDataset sanity check OK")

full_ds = SlotCacheDataset(CACHE_DIR, TRAIN_SHARDS, label_table)

In [ ]:
import subprocess
subprocess.run(["pip", "install", "-q", "timm"], check=True)
import timm
print("timm:", timm.__version__)

## Model: DINOv2-small backbone + masked_finding_attention

Identical to 05v2 - hand-kept copy of `src/model.py`'s functions.

In [ ]:
def masked_finding_attention(embeddings, mask, query, head_weight, head_bias):
    if not (mask.sum(dim=1) > 0).all():
        raise ValueError("masked_finding_attention: a row has 0 present slots")
    scores = torch.einsum("od,bsd->bos", query, embeddings) / (embeddings.shape[-1] ** 0.5)
    expanded_mask = mask.unsqueeze(1).expand(-1, query.shape[0], -1)
    scores = scores.masked_fill(expanded_mask == 0, float("-inf"))
    weights = torch.softmax(scores, dim=-1)
    context = torch.einsum("bos,bsd->bod", weights, embeddings)
    logits = (context * head_weight.unsqueeze(0)).sum(-1) + head_bias
    if torch.isnan(logits).any() or torch.isinf(logits).any():
        raise RuntimeError("masked_finding_attention produced NaN/Inf logits")
    return logits


class SlotAttentionModel(nn.Module):
    def __init__(self, n_findings=len(FINDINGS), n_slots=len(SLOT_NAMES),
                 backbone_name="vit_small_patch14_dinov2.lvd142m", unfreeze_last=6):
        super().__init__()
        self.backbone = timm.create_model(
            backbone_name, pretrained=True, num_classes=0, img_size=224,
        )
        embed_dim = self.backbone.num_features
        for p in self.backbone.parameters():
            p.requires_grad = False
        for block in self.backbone.blocks[-unfreeze_last:]:
            for p in block.parameters():
                p.requires_grad = True

        self.query = nn.Parameter(torch.randn(n_findings, embed_dim) * (embed_dim ** -0.5))
        self.heads = nn.Linear(embed_dim, n_findings)
        self.embed_dim = embed_dim
        self.n_findings = n_findings
        self.n_slots = n_slots

    def forward(self, slot_images, slot_mask):
        B, S, C, H, W = slot_images.shape
        if (S, C, H, W) != (self.n_slots, 3, 224, 224):
            raise ValueError(
                f"expected slot_images (*, {self.n_slots}, 3, 224, 224), got {tuple(slot_images.shape)}"
            )
        if tuple(slot_mask.shape) != (B, S):
            raise ValueError(f"expected slot_mask ({B}, {S}), got {tuple(slot_mask.shape)}")

        flat = slot_images.view(B * S, C, H, W)
        embeddings = self.backbone(flat).view(B, S, self.embed_dim)
        return masked_finding_attention(
            embeddings, slot_mask, self.query, self.heads.weight, self.heads.bias
        )


print("SlotAttentionModel defined - instantiating to confirm it loads real DINOv2 weights...")
_smoke_model = SlotAttentionModel()
n_trainable = sum(p.numel() for p in _smoke_model.parameters() if p.requires_grad)
n_total = sum(p.numel() for p in _smoke_model.parameters())
print(f"embed_dim={_smoke_model.embed_dim}, trainable params={n_trainable:,} / {n_total:,}")
del _smoke_model

## Evaluation helpers

Identical to 05v2 - hand-kept copies of `src/evaluate.py`.

In [ ]:
def per_finding_roc_auc(y_true, y_pred):
    scores = {}
    for c in y_true.columns:
        if y_true[c].nunique() < 2:
            scores[c] = float("nan")
        else:
            scores[c] = roc_auc_score(y_true[c], y_pred[c])
    return pd.Series(scores)


def macro_roc_auc(y_true, y_pred):
    per_finding = per_finding_roc_auc(y_true, y_pred)
    undefined = per_finding[per_finding.isna()]
    if len(undefined) > 0:
        print(f"  (macro_roc_auc: {len(undefined)} finding(s) undefined this fold "
              f"- {list(undefined.index)}, excluded from the mean, not treated as 0)")
    return float(per_finding.mean())

## Pre-flight: overfit 8 real studies

Run once (not per fold) - same architecture/wiring as 05v2, already
validated there; this just confirms nothing regressed (timm/torch
version drift, a copy-paste error in this notebook) before spending
~2.5h on 3 real training runs.

In [ ]:
model = SlotAttentionModel().to(DEVICE)
opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad],
                         lr=1e-3, weight_decay=0.02)

tiny_studies = label_table.index[:8]
tiny_ds = SlotCacheDataset(CACHE_DIR, TRAIN_SHARDS, label_table, study_ids=tiny_studies)
tiny_loader = torch.utils.data.DataLoader(tiny_ds, batch_size=8, shuffle=False)
images, mask, labels = next(iter(tiny_loader))
images, mask, labels = images.to(DEVICE), mask.to(DEVICE), labels.to(DEVICE)

eps = 1e-7
loss_floor = -(labels * torch.log(labels.clamp(eps, 1)) +
               (1 - labels) * torch.log((1 - labels).clamp(eps, 1))).mean().item()
print(f"loss floor for this batch (soft-label entropy): {loss_floor:.4f}")

print("pre-flight: overfitting 8 real studies...")
model.train()
for step in range(500):
    opt.zero_grad()
    loss = F.binary_cross_entropy_with_logits(model(images, mask), labels)
    loss.backward()
    opt.step()
    if step % 100 == 0:
        print(f"  step {step}: loss={loss.item():.4f}")
print(f"final pre-flight loss: {loss.item():.4f} (floor: {loss_floor:.4f})")
assert loss.item() < loss_floor + 0.02, (
    f"model failed to reach the soft-label loss floor ({loss_floor:.4f}) on 8 real "
    f"studies - stop and debug before the real runs"
)
print("pre-flight OK")
del model, opt

## Training function

Refactors 05v2's single-fold training loop into a reusable function, so
folds 1-3 can each call it without copy-pasting the loop 3 times.
Checkpoints to `/kaggle/working/a2_v1_fold{fold_id}_best.pt` and returns
that fold's val predictions (all studies, gold and weak) plus the
val label table, for pooling later.

In [ ]:
def train_fold(fold_id, epochs=EPOCHS, batch_size=BATCH_SIZE):
    train_idx = np.flatnonzero(label_table["fold"].to_numpy() != fold_id)
    val_idx = np.flatnonzero(label_table["fold"].to_numpy() == fold_id)
    val_labels = label_table.iloc[val_idx].reset_index()
    val_is_gold = val_labels["is_gold"].to_numpy()
    print(f"fold {fold_id}: {len(train_idx)} train / {len(val_idx)} val ({val_is_gold.sum()} gold in val)")

    train_loader = torch.utils.data.DataLoader(
        torch.utils.data.Subset(full_ds, train_idx.tolist()), batch_size=batch_size, shuffle=True, num_workers=2,
    )
    val_loader = torch.utils.data.DataLoader(
        torch.utils.data.Subset(full_ds, val_idx.tolist()), batch_size=batch_size, shuffle=False, num_workers=2,
    )

    model = SlotAttentionModel().to(DEVICE)
    backbone_params = [p for n, p in model.named_parameters() if p.requires_grad and n.startswith("backbone")]
    head_params = [p for n, p in model.named_parameters() if not n.startswith("backbone")]
    opt = torch.optim.AdamW([
        {"params": backbone_params, "lr": 8e-6},
        {"params": head_params, "lr": 1e-3},
    ], weight_decay=0.02)

    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        opt, max_lr=[8e-6, 1e-3], total_steps=epochs * len(train_loader)
    )

    ckpt_path = f"/kaggle/working/a2_v1_fold{fold_id}_best.pt"
    best_gold_auc = -1.0
    best_val_pred = None
    for epoch in range(epochs):
        model.train()
        t0 = time.time()
        for images, mask, labels in train_loader:
            images, mask, labels = images.to(DEVICE), mask.to(DEVICE), labels.to(DEVICE)
            opt.zero_grad()
            loss = F.binary_cross_entropy_with_logits(model(images, mask), labels)
            loss.backward()
            opt.step()
            scheduler.step()

        model.eval()
        probs = []
        with torch.no_grad():
            for images, mask, _ in val_loader:
                images, mask = images.to(DEVICE), mask.to(DEVICE)
                probs.append(torch.sigmoid(model(images, mask)).cpu().numpy())
        val_pred = pd.DataFrame(np.concatenate(probs), columns=FINDINGS)
        gold_auc = macro_roc_auc(val_labels.loc[val_is_gold, FINDINGS], val_pred[val_is_gold])
        print(f"  epoch {epoch}: {time.time() - t0:.0f}s, val gold macro-AUC={gold_auc:.4f}")

        if gold_auc > best_gold_auc:
            best_gold_auc = gold_auc
            best_val_pred = val_pred.copy()
            torch.save(model.state_dict(), ckpt_path)

    print(f"fold {fold_id}: best gold macro-AUC={best_gold_auc:.4f}, checkpoint saved to {ckpt_path}")
    del model, opt
    return val_labels, best_val_pred, best_gold_auc

## Fold 0: reuse the existing checkpoint (inference only)

No training - loads `a2_v1_fold0_best.pt` (trained in 05v2, on fold 0
held out) and scores it once on fold 0's val set.

In [ ]:
fold0_train_idx = np.flatnonzero(label_table["fold"].to_numpy() != 0)
fold0_val_idx = np.flatnonzero(label_table["fold"].to_numpy() == 0)
fold0_val_labels = label_table.iloc[fold0_val_idx].reset_index()
fold0_val_is_gold = fold0_val_labels["is_gold"].to_numpy()

fold0_val_loader = torch.utils.data.DataLoader(
    torch.utils.data.Subset(full_ds, fold0_val_idx.tolist()), batch_size=BATCH_SIZE, shuffle=False, num_workers=2,
)

model = SlotAttentionModel().to(DEVICE)
model.load_state_dict(torch.load(CHECKPOINT_PATH, map_location=DEVICE))
model.eval()
probs = []
with torch.no_grad():
    for images, mask, _ in fold0_val_loader:
        images, mask = images.to(DEVICE), mask.to(DEVICE)
        probs.append(torch.sigmoid(model(images, mask)).cpu().numpy())
fold0_val_pred = pd.DataFrame(np.concatenate(probs), columns=FINDINGS)
del model

fold0_gold_auc = macro_roc_auc(fold0_val_labels.loc[fold0_val_is_gold, FINDINGS], fold0_val_pred[fold0_val_is_gold])
print(f"fold 0 (reused checkpoint): gold macro-AUC={fold0_gold_auc:.4f}")
assert abs(fold0_gold_auc - 0.7689) < 0.01, (
    f"fold 0's reused checkpoint scored {fold0_gold_auc:.4f}, expected ~0.7689 - "
    f"check CHECKPOINT_PATH points at the right file and the fold split matches 05v2"
)
print("matches 05v2's recorded 0.7689 - checkpoint reuse is valid")

fold_results = {0: (fold0_val_labels, fold0_val_pred, fold0_gold_auc)}

## Folds 1, 2, 3: train fresh

Same architecture/hyperparameters as fold 0, ~47 min each.

In [ ]:
for fold_id in [1, 2, 3]:
    fold_results[fold_id] = train_fold(fold_id)

## Pooled OOF report: all 4 folds, all 58 gold studies

Concatenates every fold's gold predictions (each study appears in
exactly one fold's val set) into one 58-row table and scores the real
primary metric on it - the number this project should treat as the
trustworthy A2 v1 baseline going forward, not the single-fold 0.7689.

In [ ]:
gold_true_parts, gold_pred_parts = [], []
for fold_id in sorted(fold_results):
    val_labels, val_pred, _ = fold_results[fold_id]
    is_gold = val_labels["is_gold"].to_numpy()
    gold_true_parts.append(val_labels.loc[is_gold, FINDINGS].reset_index(drop=True))
    gold_pred_parts.append(val_pred[is_gold].reset_index(drop=True))

pooled_true = pd.concat(gold_true_parts, ignore_index=True)
pooled_pred = pd.concat(gold_pred_parts, ignore_index=True)
print(f"pooled gold studies: {len(pooled_true)} (expected 58)")
assert len(pooled_true) == 58

pooled_per_finding = per_finding_roc_auc(pooled_true, pooled_pred)
print("\npooled per-finding AUC (all 58 gold, 4-fold OOF):\n", pooled_per_finding)
pooled_macro = float(pooled_per_finding.mean())
print(f"\npooled gold macro-AUC (4-fold OOF): {pooled_macro:.4f}")

print("\nfold-by-fold gold macro-AUC (for reference, noisier - 11-17 gold each):")
for fold_id in sorted(fold_results):
    print(f"  fold {fold_id}: {fold_results[fold_id][2]:.4f}")

print(f"\ncomparison: single-fold-0 result was 0.7689 (17 gold studies).")
print(f"medial_meniscus_tear was 0.458 on fold 0 alone - pooled: {pooled_per_finding['medial_meniscus_tear']:.4f}")
print(f"oa_lateral_compartment was undefined on fold 0 alone - pooled: {pooled_per_finding['oa_lateral_compartment']:.4f}")

## Real output (pending a Kaggle run)